In [ ]:
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
!tar -xvzf images.tar.gz && tar -xvzf annotations.tar.gz
!rm images/*.mat

In [ ]:
!pip install albumentations torchmetrics pytorch_lightning -U -q

In [ ]:
import os
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchmetrics
from albumentations.pytorch.transforms import ToTensorV2
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
def preprocess_mask(mask):
    mask = mask.astype(np.float32)
    mask[mask == 2.0] = 0.0
    mask[(mask == 1.0) | (mask == 3.0)] = 1.0
    return mask

class PetDataset(Dataset):
    def __init__(self, split="train", transform=None):
        images_train, images_test = train_test_split(
            os.listdir("images"), random_state=142, shuffle=True, train_size=0.8
        )
        if split == "train":
            self.images_filenames = images_train
        else:
            self.images_filenames = images_test
        self.transform = transform

    def __len__(self):
        return len(self.images_filenames)

    def __getitem__(self, idx):
        image_filename = self.images_filenames[idx]
        image = cv2.imread(os.path.join("images", image_filename))
        if image is None:
            return self.__getitem__(idx + 1 if self.__len__() <= idx + 1 else 0)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(
            os.path.join("annotations", "trimaps", image_filename.replace(".jpg", ".png")),
            cv2.IMREAD_UNCHANGED,
        )
        if mask is None:
            return self.__getitem__(idx + 1 if self.__len__() <= idx + 1 else 0)
        mask = preprocess_mask(mask)
        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        return image, mask

train_dataset = PetDataset()
val_dataset = PetDataset(split="val")
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

In [ ]:
def display_few_examples_from_data(dataset, n=4):
    figure, ax = plt.subplots(nrows=n, ncols=2, figsize=(10, 24))
    for i in range(n):
        image, mask = dataset.__getitem__(i)
        ax[i, 0].imshow(image)
        ax[i, 1].imshow(mask, interpolation="nearest")
        ax[i, 0].set_title("Image")
        ax[i, 1].set_title("Mask")
        ax[i, 0].set_axis_off()
        ax[i, 1].set_axis_off()
    plt.tight_layout()
    plt.show()

display_few_examples_from_data(train_dataset)
print("Validation dataset")
display_few_examples_from_data(val_dataset)

In [ ]:
!pip install segmentation-models-pytorch -q

In [ ]:
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

train_dataset = PetDataset(split="train", transform=train_transform)
val_dataset = PetDataset(split="val", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
import segmentation_models_pytorch as smp

class PetSegmentationModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = smp.Unet(
            encoder_name='resnet34',
            encoder_weights='imagenet',
            in_channels=3,
            classes=1,
        )
        self.loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
        self.iou_metric = torchmetrics.JaccardIndex(task='binary')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks = batch
        masks = masks.unsqueeze(1).float()
        logits = self(images)
        loss = self.loss_fn(logits, masks)
        preds = (torch.sigmoid(logits) > 0.5).float()
        iou = self.iou_metric(preds, masks.long())
        self.log('loss/train', loss, on_epoch=True, prog_bar=True)
        self.log('iou/train', iou, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, masks = batch
        masks = masks.unsqueeze(1).float()
        logits = self(images)
        loss = self.loss_fn(logits, masks)
        preds = (torch.sigmoid(logits) > 0.5).float()
        iou = self.iou_metric(preds, masks.long())
        self.log('loss/val', loss, on_epoch=True, prog_bar=True)
        self.log('iou/val', iou, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

In [ ]:
model = PetSegmentationModel()

trainer = pl.Trainer(
    accelerator='gpu',
    max_epochs=15,
    callbacks=[
        EarlyStopping(monitor='iou/val', patience=5, mode='max'),
        ModelCheckpoint(monitor='iou/val', mode='max', save_top_k=1),
    ]
)

trainer.fit(model, train_loader, val_loader)

In [ ]:
model.eval()
model = model.cuda()

images, masks = next(iter(val_loader))
with torch.no_grad():
    preds = torch.sigmoid(model(images.cuda())).cpu()
    preds = (preds > 0.5).float().squeeze(1)

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for i in range(8):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    axes[0, i].imshow(img)
    axes[0, i].set_title('Image')
    axes[0, i].axis('off')
    axes[1, i].imshow(masks[i], cmap='gray')
    axes[1, i].set_title('True')
    axes[1, i].axis('off')
    axes[2, i].imshow(preds[i], cmap='gray')
    axes[2, i].set_title('Predicted')
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

In [7]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))

/content/drive/MyDrive/Colab Notebooks/Copy of Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Python basics.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
